# Ch 28. データ並列 — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: 4-GPU DDPシミュレーションで各GPUが異なるデータを処理した後、勾配が同一になることを示せ。

### 解法

各rankのlocal gradient $g_r$を計算後、All-Reduce平均$\bar g=N^{-1}\sum_r g_r$を全replicaへ配るため、同期直後のgradient tensorは要素ごとに同一となる。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import torch
g=torch.Generator().manual_seed(28); w=torch.randn(3,requires_grad=True,generator=g); batches=[]; local=[]
for _ in range(4):
    x=torch.randn(5,3,generator=g); y=torch.randn(5,generator=g); batches.append((x,y))
    local.append(torch.autograd.grad(((x@w-y)**2).mean(),w)[0])
allreduce_average=torch.stack(local).mean(0)
full_x=torch.cat([x for x,_ in batches]); full_y=torch.cat([y for _,y in batches])
single_process_reference=torch.autograd.grad(((full_x@w-full_y)**2).mean(),w)[0]
max_error=float((allreduce_average-single_process_reference).abs().max())
assert torch.allclose(allreduce_average,single_process_reference,atol=1e-6,rtol=1e-6)
{'allreduce_average':allreduce_average,'full_batch_reference':single_process_reference,'max_abs_error':max_error}


## 問題 2

**問題**: 有効バッチサイズ32、64、128、256で学習し収束を比較せよ。

### 解法

batch sizeだけを変え、総example数・optimizer・seed・epochを固定する。同じvalidation setのstep別lossでgradient noiseとupdate回数効果を分離する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
rng=np.random.default_rng(2); true=np.array([2.,-1.]); X=rng.normal(size=(256,2)); y=X@true
def train(bs):
 w=np.zeros(2)
 for _ in range(20):
  for i in range(0,256,bs): w-=.1*(2/bs)*X[i:i+bs].T@(X[i:i+bs]@w-y[i:i+bs])
 return np.mean((X@w-y)**2)
loss={b:train(b) for b in [32,64,128,256]}; assert all(np.isfinite(list(loss.values()))); loss


## 問題 3

**問題**: All-Reduceの通信量が$O(\|\theta\|)$となる理由を説明せよ。

### 解法

Ring All-Reduceはsize $P=|\theta|$のtensorをreduce-scatterとall-gatherで各約$(N-1)P/N$送る。合計は$2P$未満なのでrank当たり$O(P)$である。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
P=10_000_000; workers=[2,4,8]; volume={n:2*(n-1)/n*P for n in workers}
assert all(v<2*P for v in volume.values()); volume


## 問題 4

**問題**: 通信と計算のオーバーラップが速度を向上させる理由を説明せよ。

### 解法

直列時間は$C+M$、完全overlapの下限は$\max(C,M)$。backward bucket準備直後に非同期通信を開始しcritical pathから一部通信を隠す。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
compute,comm=8.,5.; serial=compute+comm; overlapped=max(compute,comm)
assert overlapped<serial; {'serial':serial,'overlapped_ideal':overlapped,'speedup':serial/overlapped}


## 問題 5

**問題**: LARSが大規模バッチで有効な理由を説明せよ。

### 解法

大規模batchでは層ごとのweight norm対gradient norm比が大きく異なる。LARSは$\eta_l=\eta\|w_l\|/(\|g_l\|+\lambda\|w_l\|)$で各層の相対更新量を制御する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
theta=np.array([100.,1.]); grad=np.array([1.,1.]); eta=.1
trust=eta*np.linalg.norm(theta)/(np.linalg.norm(grad)+1e-9)
assert trust>eta; {'global_lr':eta,'layer_scaled_lr':trust}
